# Historical daily data EDA

This is a self-contained EDA notebook for the historical main and crypto snapshots. It reads only the CSV and JSON outputs created by the export script.

Set `USE_FILTERED = True` to inspect the both-filter view instead of the full union view.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
plt.style.use("seaborn-v0_8-whitegrid")

project_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in project_roots
     if (path / "data/historical_2020_2024").is_dir()),
    Path.cwd(),
)
DATA_DIR = PROJECT_ROOT / "data" / "historical_2020_2024"
USE_FILTERED = False
GROUPS = ["main", "crypto"]

In [ ]:
metadata_text_columns = [
    "market_id", "market_ticker", "event_ticker", "series_ticker",
    "market_question", "market_subtitle", "yes_subtitle",
    "no_subtitle", "market_rules", "event_question",
    "event_subtitle", "series_title", "series_category",
    "series_frequency", "market_status", "market_result",
    "download_status",
]

metadata_frames = []
for group in GROUPS:
    directory = DATA_DIR / (f"{group}_filtered" if USE_FILTERED else group)
    frame = pd.read_csv(
        directory / f"{group}_market_metadata.csv",
        dtype={column: "string" for column in metadata_text_columns},
    )
    frame["group"] = group
    metadata_frames.append(frame)

metadata_all = pd.concat(metadata_frames, ignore_index=True)
metadata_all[["group", "market_id", "market_question"]].head(10)

In [ ]:
candles_frames = []
summaries = {}

for group in GROUPS:
    directory = DATA_DIR / (f"{group}_filtered" if USE_FILTERED else group)
    candles = pd.read_csv(
        directory / f"{group}_daily_candles.csv",
        dtype={
            "market_id": "string",
            "market_ticker": "string",
            "series_ticker": "string",
        },
    )
    candles["group"] = group
    candles_frames.append(candles)
    with (directory / "export_summary.json").open("r", encoding="utf-8") as handle:
        summaries[group] = json.load(handle)

candles_all = pd.concat(candles_frames, ignore_index=True)
candles_all[["group", "market_id", "date_utc", "price_mean", "volume"]].head(10)

In [ ]:
for column in [
    "history_start_ts", "history_end_ts",
    "first_observation_ts", "last_observation_ts",
]:
    metadata_all[column] = pd.to_datetime(
        pd.to_numeric(metadata_all[column], errors="coerce"),
        unit="s", utc=True, errors="coerce",
    )

for column in [
    "passes_volume_filter", "passes_lifetime_filter",
    "passes_both_filters", "volume_fp", "lifetime_days",
    "expected_daily_rows", "received_daily_rows",
    "actual_candles_exported",
]:
    metadata_all[column] = pd.to_numeric(metadata_all[column], errors="coerce")

for column in ["open_time", "close_time"]:
    metadata_all[column] = pd.to_datetime(
        metadata_all[column], utc=True, errors="coerce"
    )

candles_all["end_period_ts"] = pd.to_numeric(
    candles_all["end_period_ts"], errors="coerce"
)
candles_all["date_utc"] = pd.to_datetime(
    candles_all["date_utc"], utc=True, errors="coerce"
)
for column in [
    "price_open", "price_low", "price_high", "price_close",
    "price_mean", "price_previous", "yes_bid_close",
    "yes_ask_close", "volume", "open_interest",
]:
    candles_all[column] = pd.to_numeric(candles_all[column], errors="coerce")

candles_all[["date_utc", "price_mean", "price_low", "price_high"]].describe()

In [ ]:
status_counts = (
    metadata_all.groupby(["group", "download_status"], dropna=False)
    .size()
    .reset_index(name="markets")
    .sort_values(["group", "markets"], ascending=[True, False])
)
display(status_counts)

In [ ]:
metadata = metadata_all.loc[
    metadata_all["download_status"].eq("success")
].reset_index(drop=True)
candles = candles_all.loc[
    candles_all["market_id"].isin(metadata["market_id"])
].reset_index(drop=True)

main_metacols = [
    "group", "market_id", "market_question", "volume_fp",
    "lifetime_days", "market_result",
]
metadata[main_metacols].head(10)

## 1. Snapshot overview

We first check how large each group is and which dates the exported snapshot actually covers.

In [ ]:
overview = (
    metadata.groupby("group")
    .agg(
        markets=("market_id", "nunique"),
        events=("event_ticker", "nunique"),
        series=("series_ticker", "nunique"),
        candle_rows=("actual_candles_exported", "sum"),
        first_market_open=("open_time", "min"),
        last_market_close=("close_time", "max"),
    )
    .reset_index()
)
overview["source_mode"] = overview["group"].map(
    lambda group: summaries[group].get("source_mode")
)
display(overview)

## 2. Calendar coverage

These views show whether activity is spread across the historical period or concentrated in particular dates.

In [ ]:
daily_summary = (
    candles.groupby(["date_utc", "group"])
    .agg(
        markets=("market_id", "nunique"),
        candle_rows=("market_id", "size"),
        price_close_valid=("price_close", "count"),
        bid_close_valid=("yes_bid_close", "count"),
        ask_close_valid=("yes_ask_close", "count"),
        total_volume=("volume", "sum"),
        mean_open_interest=("open_interest", "mean"),
    )
    .reset_index()
    .sort_values(["date_utc", "group"])
)
daily_summary.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for group, rows in daily_summary.groupby("group"):
    rows.plot(
        x="date_utc", y="markets", ax=ax,
        linewidth=1.5, label=group,
    )
ax.set_xlabel("UTC date")
ax.set_ylabel("markets")
ax.set_title("Distinct markets represented by UTC date")
ax.legend(title="group")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for group, rows in daily_summary.groupby("group"):
    rows.plot(
        x="date_utc", y="candle_rows", ax=ax,
        linewidth=1.5, label=group,
    )
ax.set_xlabel("UTC date")
ax.set_ylabel("candle rows")
ax.set_title("Daily candle rows by UTC date")
ax.legend(title="group")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for group, rows in daily_summary.groupby("group"):
    rows.plot(
        x="date_utc", y="total_volume", ax=ax,
        linewidth=1.5, label=group,
    )
ax.set_xlabel("UTC date")
ax.set_ylabel("contracts traded")
ax.set_title("Daily traded volume by UTC date")
ax.legend(title="group")
plt.tight_layout()
plt.show()

## 3. Market dates and question context

This table connects each observed path to its question, category, lifecycle, and realized result.

In [ ]:
market_activity = (
    candles.groupby(["market_id", "group"])
    .agg(
        observed_days=("date_utc", "nunique"),
        price_close_days=("price_close", "count"),
        total_volume=("volume", "sum"),
        first_candle=("date_utc", "min"),
        last_candle=("date_utc", "max"),
    )
    .reset_index()
)

question_context = (
    metadata.merge(
        market_activity, on=["market_id", "group"],
        how="inner", validate="one_to_one",
    )
    .sort_values(["price_close_days", "total_volume"], ascending=False)
    [[
        "group", "market_id", "market_question",
        "event_question", "series_title", "series_category",
        "open_time", "close_time", "first_candle",
        "last_candle", "observed_days", "price_close_days",
        "total_volume", "market_result",
    ]]
    .head(10)
)
display(question_context)

In [ ]:
def plot_price_mean_for_market(market_id, ax=None):
    market = metadata.loc[metadata["market_id"].eq(market_id)].iloc[0]
    sample = candles.loc[candles["market_id"].eq(market_id)].sort_values("date_utc")
    question = (
        market["market_question"]
        if pd.notna(market["market_question"])
        else "question unavailable"
    )
    result = (
        market["market_result"]
        if pd.notna(market["market_result"])
        else "unknown"
    )
    volume = market["volume_fp"]
    volume_label = f"{volume:,.0f}" if pd.notna(volume) else "n/a"

    if ax is None:
        _, ax = plt.subplots(figsize=(14, 5))
    ax.plot(sample["date_utc"], sample["price_mean"], marker="o",
            linewidth=2, label="price_mean")
    ax.plot(sample["date_utc"], sample["price_close"],
            linestyle="--", alpha=0.75, label="price_close")
    price_range = sample.dropna(subset=["price_low", "price_high"])
    if not price_range.empty:
        ax.fill_between(
            price_range["date_utc"], price_range["price_low"],
            price_range["price_high"], alpha=0.18,
            label="price_low–price_high",
        )
    ax.set_ylim(0, 1)
    ax.set_ylabel("probability / price")
    ax.set_title(
        f"[{market['group']}] {question}\n"
        f"Outcome={result} | volume={volume_label} contracts"
    )
    ax.legend(loc="best")
    return ax

## 4. Example probability paths

These examples show the daily probability path for the most persistently observed markets. They are descriptive, not a forecast evaluation.

In [ ]:
sample_ids = (
    market_activity.loc[market_activity["price_close_days"].gt(0)]
    .sort_values(["price_close_days", "total_volume"], ascending=False)
    .head(3)["market_id"]
    .tolist()
)

fig, axes = plt.subplots(
    len(sample_ids), 1, figsize=(14, 3.5 * len(sample_ids)), squeeze=False
)
for row_index, market_id in enumerate(sample_ids):
    plot_price_mean_for_market(market_id, ax=axes[row_index, 0])
axes[-1, 0].set_xlabel("UTC date")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 5. Group comparison

This compact table describes how the two groups differ in size, resolution, activity, and observed coverage.

In [ ]:
group_comparison = (
    metadata.groupby("group")
    .agg(
        markets=("market_id", "nunique"),
        resolved_markets=("market_result", lambda values: int(values.isin(["yes", "no"]).sum())),
        mean_volume_contracts=("volume_fp", "mean"),
        median_lifetime_days=("lifetime_days", "median"),
        mean_observed_candles=("actual_candles_exported", "mean"),
        first_observed_date=("first_observation_ts", "min"),
        last_observed_date=("last_observation_ts", "max"),
    )
    .reset_index()
)
display(group_comparison)

## 6. Final quality checks

These checks confirm that the combined files are structurally ready for outcome alignment and forecast evaluation.

In [ ]:
quality = pd.DataFrame([
    {"check": "market_id unique in metadata", "value": bool(metadata["market_id"].is_unique)},
    {"check": "market-day unique in candles", "value": bool(not candles.duplicated(["market_id", "end_period_ts"]).any())},
    {"check": "all candle markets have metadata", "value": bool(candles["market_id"].isin(metadata["market_id"]).all())},
    {"check": "all source modes are historical", "value": all(summary.get("source_mode") == "historical" for summary in summaries.values())},
    {"check": "no completely empty candle fields", "value": bool(not candles.isna().all(axis=0).any())},
    {"check": "price_close coverage", "value": f"{candles['price_close'].notna().mean():.1%}"},
    {"check": "price_low/high interval coverage", "value": f"{candles[['price_low', 'price_high']].notna().all(axis=1).mean():.1%}"},
    {"check": "yes bid/ask close coverage", "value": f"{candles[['yes_bid_close', 'yes_ask_close']].notna().all(axis=1).mean():.1%}"},
])
display(quality)

## Next step: outcome-based research

The combined snapshot is ready for outcome alignment, forecast timing rules, Brier Score calculations, calibration plots, and comparisons between main and crypto.